# 99 - Synthese: Was kann das BMDS steuern?

**Die vier Befunde stehen nicht beziehungslos nebeneinander. Sie ergeben zusammen eine strukturelle Diagnose: zentrale Steuerung ist auf einen begrenzten Ausschnitt des Digitalhaushalts beschränkt, und ein Digitalbudget muss zwei unterschiedliche Hoheiten auseinanderhalten, die im Diskurs vermischt werden.**

Essay-Bezug: Abschnitt 6 (Akt 4), Abschnitt 7 (Zusammenführung), Abschnitt 8 (Limitationen), Abschnitt 9 (Daten, Code, Reproduzierbarkeit), Quellen

## Die vier Akte im Überblick

| Akt | Kernsatz | Hauptbefund |
|---|---|---|
| **1 - Wo liegt das Geld?** (H1, Notebook 01) | Polyzentrik statt Schatten-Digitalministerium | HHI 2024 = 1.612 (moderat fragmentiert), sechs Ressorts mit fundamental verschiedenen Aufgaben-Profilen decken 92 % ab; realistische BMDS-Bündelung ca. 21–23 % |
| **2 - Was passiert damit?** (H2, Notebook 02) | Transfer statt Aufbau | Eigene Investitionen stagnieren bei rund einer Milliarde Euro (10,9 → 6,0 % Anteil), Transfer-Investitionen verdoppeln ihren Anteil (15,9 → 31,8 %), getrieben von Breitband/ETCS (BMDV) und DigitalPakt-Folge (BMBF) |
| **3 - Wie wird darüber geredet?** (H4, Notebook 03) | Rebranding statt Buzzword-Inflation | Generische Begriffe wachsen volumetrisch, nicht in Treffer-Häufigkeit; die Hightech-Strategie verschwindet vollständig (40 Treffer / 1,32 Mrd. € → 0/0), die Zukunftsstrategie erscheint bei vergleichbarem Volumen neu (0/0 → 46 / 1,89 Mrd. €) |
| **4 - Wer redet worüber?** (H5, Notebook 04) | Aufmerksamkeit und Geld sind nicht deckungsgleich | BMDV: Volumen-Rang 1, Aufmerksamkeits-Rang 4 (Diskrepanz +3); BMI: Volumen-Rang 5, Aufmerksamkeits-Rang 1 (Diskrepanz −4); BMBF und BMVg: Match (Diskrepanz 0) |

**Verbindung der Akte.** Akt 1 setzt die Grenze für zentrale Steuerung (ein Fünftel des Volumens). Akt 2 zeigt, dass selbst innerhalb dieser Grenze der Bund seine Rolle vom Selbst-Aufbau zur Verteilung verschiebt. Akt 3 zeigt, dass die politische Sprache auf Regierungswechsel reagiert, der inhaltliche Kern aber stabil bleibt. Akt 4 fügt eine dritte Achse hinzu: Aufmerksamkeit und Mittel sind entkoppelt, das BMDV regiert das Geld, das BMI regiert die Schlagzeilen des Bundestages.

## Methodik dieser Zusammenführung

Dieses Notebook wiederholt keine Analyse, es trägt die zentralen Kennzahlen aus den Notebooks 01–04 in einer kompakten Tabelle zusammen, jeweils für das aktuellste verfügbare Jahr (2024) und im Vergleich zum Ausgangsjahr (2019), soweit anwendbar. Grundlage bleibt durchgehend `digi_soll_eng`. Die Zahlen selbst sind an anderer Stelle hergeleitet (HHI in Notebook 01, Eigen-/Transfer-Anteile in Notebook 02, Tag-Frequenzen und -Volumina in Notebook 03, Aufmerksamkeits-Anteil in Notebook 04); hier geht es um den Zusammenhang, nicht um neue Befunde.

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, '..')  # repo root, relativ zum notebooks/-Verzeichnis

import pandas as pd
from src.load import load, hhi

df = load()  # nutzt Repo-Root-Default aus src.load
d = df[(df['any_tag'] == 1) & (df['digi_soll_eng'] > 0)].copy()

print("=== Synthese-Tabelle: zentrale Kennzahlen je Akt (2024, ggf. vs. 2019) ===\n")

# Akt 1: Konzentration (HHI)
hhi_2019 = hhi(df, 2019)
hhi_2024 = hhi(df, 2024)
print(f"Akt 1 - HHI Ressort-Konzentration:      2019 = {hhi_2019:.0f}   2024 = {hhi_2024:.0f}")

# Akt 2: Eigene vs. Transfer-Investition (Anteil am Digital-Soll)
inv = (
    d[d['typ_invest'].notna()]
    .groupby(['jahr', 'typ_invest'])['digi_soll_eng']
    .sum().div(1e6).unstack().fillna(0)
)
sigma = d.groupby('jahr')['digi_soll_eng'].sum().div(1e6)
eigen_pct_2019 = inv.loc[2019, 'A_eigene_Invest'] / sigma[2019] * 100
eigen_pct_2024 = inv.loc[2024, 'A_eigene_Invest'] / sigma[2024] * 100
transfer_pct_2019 = inv.loc[2019, 'B_Transfer_Invest'] / sigma[2019] * 100
transfer_pct_2024 = inv.loc[2024, 'B_Transfer_Invest'] / sigma[2024] * 100
print(f"Akt 2 - Anteil eigene Investition:       2019 = {eigen_pct_2019:.1f} %   2024 = {eigen_pct_2024:.1f} %")
print(f"Akt 2 - Anteil Transfer-Investition:     2019 = {transfer_pct_2019:.1f} %   2024 = {transfer_pct_2024:.1f} %")

# Akt 3: Rebranding-Volumen (Hightech- vs. Zukunftsstrategie)
df['tags'] = df['titel_text'].apply(
    lambda s: [] if not isinstance(s, str) else
    [t.strip().lower() for t in pd.Series(s).str.findall(r'\[([^\]]+)\]').iloc[0]]
)
for tag, jahr in [('hightech-strategie', 2019), ('hightech-strategie', 2024),
                  ('zukunftsstrategie', 2019), ('zukunftsstrategie', 2024)]:
    has_tag = df['tags'].apply(lambda lst, t=tag: t in lst)
    sub = df[has_tag & (df['jahr'] == jahr)]
    n, v = len(sub), sub['digi_soll_eng'].sum() / 1e6
    print(f"Akt 3 - {tag:<20} {jahr}:  n = {n:>3}   V = {v:.2f} Mrd. €")
# Akt 4: Aufmerksamkeits-Volumen-Diskrepanz (aus outputs/h5_panel.csv)
from src import OUTPUTS_DIR
h5 = __import__('pandas').read_csv(OUTPUTS_DIR / 'h5_panel.csv')
h5_24 = h5[(h5['jahr'] == 2024) & (h5['ressort'] != 'EP60')].copy()
h5_24['rang_vol'] = h5_24['mrd_eur'].rank(ascending=False)
h5_24['rang_aufm'] = h5_24['reden_anteil_promille'].rank(ascending=False)
h5_24['diskrepanz'] = h5_24['rang_aufm'] - h5_24['rang_vol']
top_buzz = h5_24.loc[h5_24['diskrepanz'].idxmin(), 'ressort']
top_still = h5_24.loc[h5_24['diskrepanz'].idxmax(), 'ressort']
max_disk_neg = h5_24['diskrepanz'].min()
max_disk_pos = h5_24['diskrepanz'].max()
bmdv_promille = h5_24.loc[h5_24['ressort'] == 'BMDV', 'reden_anteil_promille'].values[0]
bmi_promille = h5_24.loc[h5_24['ressort'] == 'BMI', 'reden_anteil_promille'].values[0]
print(f"Akt 4 - Top Buzz-Ressort 2024:          {top_buzz} (Diskrepanz {max_disk_neg:.0f})")
print(f"Akt 4 - Top stilles Schwergewicht 2024: {top_still} (Diskrepanz +{max_disk_pos:.0f})")
print(f"Akt 4 - BMDV Aufmerksamkeit:            {bmdv_promille:.2f} Promille (Median-Rang)")
print(f"Akt 4 - BMI Aufmerksamkeit:             {bmi_promille:.2f} Promille (Rang 1)")


=== Synthese-Tabelle: zentrale Kennzahlen je Akt (2024, ggf. vs. 2019) ===

Akt 1 - HHI Ressort-Konzentration:      2019 = 1701   2024 = 1612
Akt 2 - Anteil eigene Investition:       2019 = 10.9 %   2024 = 6.0 %
Akt 2 - Anteil Transfer-Investition:     2019 = 15.9 %   2024 = 31.8 %
Akt 3 - hightech-strategie   2019:  n =  40   V = 1.32 Mrd. €
Akt 3 - hightech-strategie   2024:  n =   0   V = 0.00 Mrd. €
Akt 3 - zukunftsstrategie    2019:  n =   0   V = 0.00 Mrd. €
Akt 3 - zukunftsstrategie    2024:  n =  46   V = 1.89 Mrd. €
Akt 4 - Top Buzz-Ressort 2024:          BMI (Diskrepanz -4)
Akt 4 - Top stilles Schwergewicht 2024: BMDV (Diskrepanz +3)
Akt 4 - BMDV Aufmerksamkeit:            2.96 Promille (Median-Rang)
Akt 4 - BMI Aufmerksamkeit:             10.23 Promille (Rang 1)


**Befund.** Die vier Kennzahlen-Reihen zeigen denselben Grundzug aus vier Blickwinkeln: Konzentration bleibt moderat (der Herfindahl-Hirschman-Index (HHI) bewegt sich im Korridor um 1.600), die Investitionstätigkeit verschiebt sich von eigener Durchführung zu Förderung Dritter, und das politische Vokabular tauscht Etiketten bei stabilem Volumen. Keiner der vier Befunde widerspricht den anderen - sie verstärken sich zu einem Bild eines Bundes, der zunehmend koordiniert und finanziert, aber nicht zentral baut, einheitlich benennt oder in der parlamentarischen Debatte deckungsgleich repräsentiert ist.

## Politische Implikation: zwei Hoheiten trennen

Ein Digitalbudget muss zwei Bewegungen unterscheiden, die in der aktuellen Debatte vermischt werden:

1. **Klassifikations- und Transparenzhoheit.** Welche Haushaltstitel gelten als „digital"? Wer definiert eng und weit? Wer berichtet einheitlich? Diese Hoheit kann das BMDS sinnvoll und ohne große Verfassungs- oder Sachkonflikte übernehmen - sie ist der direkte Hebel für mehr Steuerbarkeit und demokratische Kontrolle.
2. **Operative Steuerungshoheit über die Mittel.** Welches Ressort entscheidet, wie viel für welche digitale Aufgabe ausgegeben wird? Diese Hoheit lässt sich aus den hier gezeigten strukturellen Gründen - Polyzentrik (Akt 1), Transfer-Charakter (Akt 2), Ressort-spezifisches Vokabular und Industriepolitik in Grauzonen wie EP 60 (Akt 3) - nur sehr begrenzt zentralisieren. Realistisch bleibt sie für Forschung, Bildung, Verkehrsinfrastruktur, Wehr-IT und Industriepolitik bei den Fachressorts.

**Kernsatz.** Wer beides als BMDS-Mandat verspricht, verspricht zu viel. Wer aber das erste konsequent einlöst und das zweite ehrlich auf rund ein Fünftel des Digital-Solls begrenzt, kann die strukturelle Lücke schließen, die ein „Digitalbudget" als demokratischer Kontrollmechanismus tatsächlich hat.

## Limitationen und offene Fragen

Die Grenzen dieser Analyse liegen auf vier Ebenen:

- **Soll, nicht Ist.** Wir analysieren Haushaltssollansätze, nicht tatsächliche Auszahlungen.
- **Verpflichtungsermächtigungen.** Siehe Methoden-Box in Notebook 02. Soll-Rückgänge können haushaltstechnische Verschiebungen statt Politikwechsel sein.
- **Klassifikation als Setzung.** Die Klassifikation des Zentrums für Europäische Wirtschaftsforschung (ZEW) - etwa Mikroelektronik als „digital", ist eine politische Entscheidung, über die diskutiert werden kann. Ebenso die 1/n-Allokation bei Mehrfachkategorien (`kategorien_allokation()`).
- **Jahres-Lücken.** 2020 und 2022 fehlen, Corona-/Energiekrise-Effekte bleiben außen vor.
- **Externe Validierung.** Die drei großen Transfer-Posten aus Akt 2 (Breitband, ETCS, DigitalPakt-Folge) sowie der Mikroelektronik-Befund aus dem EP-60-Spin-off sind bislang nur datenseitig identifiziert. Eine qualitative Validierung über IT-Großprojekte-Bericht, Bundesrechnungshof-Bemerkungen und Bundestags-Drucksachen steht aus.
- **Akt 4: Reden-Mapping approximativ.** Das Schlagwort-Mapping ist eine Näherung; Reden besprechen Themen, nicht Haushaltspositionen. Der Drucksachen-Korpus CDRS-BT ist nur bis 2017 verfügbar und konnte nicht einbezogen werden.

## Daten, Code, Reproduzierbarkeit

Sämtliche Analysen und Charts sind im öffentlichen Repository reproduzierbar. Die zentrale Lade-Funktion (`src/load.py`) repliziert die ZEW-Eckwerte auf 0,3 Prozent genau (siehe Notebook 00). Die Notebooks 00 bis 04 sowie die Chart-Notebooks decken alle Hauptbefunde ab. Notebook 04 verschneidet den Digitalhaushalt mit dem CPP-BT-Korpus (Fobbe 2026); dafür ist der CPP-BT-ZIP unter data/external/ erforderlich.

Repository-URL: (wird nach Einreichung auf GitHub veröffentlicht)

Lizenzen: Code unter MIT, Inhalte unter CC BY 4.0.

**Quellen.**

Die nachfolgenden Angaben decken die bei Einreichung konsultierten Quellen ab. Externe Validierungsquellen werden in einer Folgeversion ergänzt.

- ZEW Mannheim / Agora Digitale Transformation (2025): Digitalhaushalt Open Data. https://agoradigital.de/projekte/digitalhaushalt
- Fobbe, Sean (2026): CPP-BT, Version 2026-01-17. Zenodo. https://doi.org/10.5281/zenodo.4542661 (Public Domain, CC0)
- ZEW/Agora-Studie zum Digitalhaushalt 2025
- Agora Digitale Transformation: Policy Paper „Vom Digitalhaushalt zum Digitalbudget"
- Agora Digitale Transformation: Analyse Haushalt 2026 (BMDS)
- Agora Digitale Transformation: Policy Paper „Der erhoffte Schub?"
- IT-Großprojekte-Bericht des Bundes (jährlich, BMI)
- Bundesrechnungshof, Bemerkungen zu IT-Vorhaben
- Bundeshaushaltspläne 2019, 2021, 2023, 2024

## Kernbefund

**Polyzentrik, Transfer-Charakter, politisches Rebranding und Aufmerksamkeits-Diskrepanz sind vier Ausdrucksformen derselben strukturellen Lage: Der Bund koordiniert und finanziert digitale Politik zunehmend, baut und benennt sie aber nicht einheitlich und redet nicht hauptsächlich über das, wofür er am meisten ausgibt.** Eine zentrale Steuerung durch ein BMDS ist deshalb nur in einem begrenzten Korridor. Klassifikation, Transparenz, Verwaltungs-IT der zivilen Ressorts, realistisch zu verorten; die operative Hoheit über das Gros der Mittel bleibt bei den Fachressorts. Diese Trennung ehrlich zu benennen, ist die eigentliche Voraussetzung für ein funktionierendes Digitalbudget.

Akt 4 (H5) ist abgeschlossen: BMDV finanziert still im Hintergrund (Volumen-Rang 1, Aufmerksamkeits-Rang 4), BMI dominiert den parlamentarischen Digitalraum weit über seinen Mittelanteil hinaus (Aufmerksamkeits-Rang 1, Volumen-Rang 5).